In [6]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ## 1. Data Loading and Preprocessing

# ### 1.1 MJO Data
mjo_file = 'ERA5 MJO (1950-2024).csv'
df_mjo = pd.read_csv(mjo_file, parse_dates=['date'])
# Filter 1960-2024
df_mjo = df_mjo[(df_mjo['date'].dt.year >= 1960) & (df_mjo['date'].dt.year <= 2024)]
# Extract month, keep June–October
df_mjo['month'] = df_mjo['date'].dt.month
df_mjo = df_mjo[df_mjo['month'].between(6, 10)]
# Keep active days with MJO amplitude ≥ 1
df_mjo = df_mjo[df_mjo['amplitude'] >= 1].copy()
# Convert phase to integer (original data may be float)
df_mjo['phase'] = df_mjo['phase'].astype(int)
# Add phase group label
def phase_group(phase):
    if phase in [1, 2]:
        return '1-2'
    elif phase in [3, 4]:
        return '3-4'
    elif phase in [5, 6]:
        return '5-6'
    elif phase in [7, 8]:
        return '7-8'
    else:
        return None
df_mjo['phase_group'] = df_mjo['phase'].apply(phase_group)
# Set date index
df_mjo.set_index('date', inplace=True)
print(f"Total MJO active days (1960-2024, Jun-Oct, amp≥1): {len(df_mjo)}")
print("Days per phase group:")
print(df_mjo['phase_group'].value_counts().sort_index())

# ### 1.2 Typhoon Landfall Information
info_file = '/mnt/home/lx/Databank/CMABSTdata/typhoon_output/landfall_typhoons_info.csv'
df_landfall = pd.read_csv(info_file, parse_dates=['landfall_time'])
# Filter years 1960-2024
df_landfall = df_landfall[(df_landfall['year'] >= 1960) & (df_landfall['year'] <= 2024)]
# Extract landfall date (keep as datetime64[ns], date part only)
df_landfall['landfall_date'] = df_landfall['landfall_time'].dt.normalize()  # time part set to 00:00:00
# Extract month and filter June-October
df_landfall['month'] = df_landfall['landfall_time'].dt.month
df_landfall = df_landfall[df_landfall['month'].between(6, 10)]
# Merge MJO information (match on landfall date)
df_landfall = df_landfall.merge(
    df_mjo[['phase', 'amplitude', 'phase_group']].reset_index(),
    left_on='landfall_date',
    right_on='date',
    how='inner'
)
# Keep landfall events with MJO amplitude ≥ 1 (already filtered in df_mjo, but ensure after merge)
df_landfall = df_landfall[df_landfall['amplitude'] >= 1]
print(f"Landfall events meeting criteria: {len(df_landfall)}")

# ### 1.3 Compute ACE per Typhoon (from track data)
track_file = '/mnt/home/lx/Databank/CMABSTdata/typhoon_output/landfall_typhoons_tracks.csv'
df_track = pd.read_csv(track_file, parse_dates=['TIME'])
# Filter years 1960-2024
df_track = df_track[(df_track['year'] >= 1960) & (df_track['year'] <= 2024)]
# Calculate ACE contribution per 6-hour record: WND^2 * 6 (unit: (m/s)^2 * hour)
# Note: WND may have missing values; those records contribute zero ACE
df_track['ACE_contrib'] = df_track['WND']**2 * 6   # 6-hour interval
# Sum by typhoon code
ace_per_typhoon = df_track.groupby('chinese_code')['ACE_contrib'].sum().to_dict()
print(f"Number of typhoons with ACE computed: {len(ace_per_typhoon)}")

# ### 1.4 Compute Total Precipitation per Typhoon (from NetCDF files)
precip_base = Path('/mnt/home/lx/storm/cma-storm2026-mjo-01/pre')
precip_per_typhoon = {}
# Typhoon codes needed (from landfall events)
typhoon_codes = df_landfall['chinese_code'].unique()
for code in typhoon_codes:
    file_path = precip_base / f"pre_{code:04d}.nc"
    if not file_path.exists():
        print(f"Warning: precipitation file {file_path} does not exist, typhoon {code} precipitation set to 0")
        precip_per_typhoon[code] = 0.0
        continue
    try:
        ds = xr.open_dataset(file_path)
        # Sum over time and space dimensions to get total precipitation
        # Note: assuming units are mm
        total_precip = ds['prec'].sum(dim=['time', 'lat', 'lon']).item()
        precip_per_typhoon[code] = total_precip
        ds.close()
    except Exception as e:
        print(f"Error reading precipitation file {file_path}: {e}, typhoon {code} precipitation set to 0")
        precip_per_typhoon[code] = 0.0
print(f"Number of typhoons with precipitation data loaded: {len(precip_per_typhoon)}")

# ## 2. Build Daily Statistics Table (MJO Active Days)


# Start from df_mjo because every MJO active day is needed
daily_stats = df_mjo[['phase_group']].copy()
daily_stats['has_landfall'] = 0
daily_stats['has_cat1'] = 0
daily_stats['has_cat2'] = 0
daily_stats['has_cat3'] = 0
daily_stats['sum_ACE_day'] = 0.0
daily_stats['sum_precip_day'] = 0.0
daily_stats['sum_precip_cat1'] = 0.0
daily_stats['sum_precip_cat2'] = 0.0
daily_stats['sum_precip_cat3'] = 0.0

# Group landfall events by date, compute daily statistics
landfall_grouped = df_landfall.groupby('landfall_date')
for date, group in landfall_grouped:
    # date is datetime64[ns] (normalized)
    if date not in daily_stats.index:
        # Should not happen because df_landfall already matched with MJO, but just in case
        continue
    daily_stats.loc[date, 'has_landfall'] = 1
    # Check which categories had a landfall
    cats = group['landfall_wind_category'].unique()
    if 1 in cats:
        daily_stats.loc[date, 'has_cat1'] = 1
    if 2 in cats:
        daily_stats.loc[date, 'has_cat2'] = 1
    if 3 in cats:
        daily_stats.loc[date, 'has_cat3'] = 1
    # Sum ACE and precipitation (also by category)
    total_ace = 0.0
    total_precip = 0.0
    precip_cat1 = 0.0
    precip_cat2 = 0.0
    precip_cat3 = 0.0
    for _, row in group.iterrows():
        code = row['chinese_code']
        cat = row['landfall_wind_category']
        total_ace += ace_per_typhoon.get(code, 0.0)
        pcp = precip_per_typhoon.get(code, 0.0)
        total_precip += pcp
        if cat == 1:
            precip_cat1 += pcp
        elif cat == 2:
            precip_cat2 += pcp
        elif cat == 3:
            precip_cat3 += pcp
    daily_stats.loc[date, 'sum_ACE_day'] = total_ace
    daily_stats.loc[date, 'sum_precip_day'] = total_precip
    daily_stats.loc[date, 'sum_precip_cat1'] = precip_cat1
    daily_stats.loc[date, 'sum_precip_cat2'] = precip_cat2
    daily_stats.loc[date, 'sum_precip_cat3'] = precip_cat3

# Days in daily_stats but not in df_landfall (has_landfall still 0) are expected.

# ## 3. Define Bootstrap Function

def bootstrap_ci_pvalue(obs_value, group_days, total_df, stat_func, n_bootstrap=1000, alpha=0.10):
    """
    Compute confidence interval and two-sided p-value via bootstrap.
    obs_value: observed statistic for the group
    group_days: number of days in the group (sample size for each bootstrap draw)
    total_df: full daily_stats DataFrame (population)
    stat_func: function that takes a DataFrame and returns the statistic of interest
    n_bootstrap: number of bootstrap samples
    alpha: significance level (0.10 => 90% CI)
    """
    boot_stats = []
    for i in range(n_bootstrap):
        sample = total_df.sample(n=group_days, replace=True, random_state=i)
        boot_stats.append(stat_func(sample))
    ci_lower = np.percentile(boot_stats, 100 * alpha/2)
    ci_upper = np.percentile(boot_stats, 100 * (1 - alpha/2))
    # Two-sided p-value
    p_value = (np.sum(np.array(boot_stats) >= obs_value) / n_bootstrap) * 2
    p_value = min(p_value, 1.0)
    significant = (obs_value < ci_lower) or (obs_value > ci_upper)
    return ci_lower, ci_upper, p_value, significant

# ## 4. Part 1: Landfall Occurrence Rate

print("\n" + "="*80)
print("Part 1: Influence of MJO Phase on Landfall Occurrence Rate")
print("="*80)

groups = ['1-2', '3-4', '5-6', '7-8']
results_rate = {}

for group in groups:
    group_dates = daily_stats[daily_stats['phase_group'] == group]
    n_days = len(group_dates)
    obs_rate = group_dates['has_landfall'].mean()
    # Statistic function: mean(has_landfall) of the sample
    stat_func = lambda df: df['has_landfall'].mean()
    ci_low, ci_high, pval, sig = bootstrap_ci_pvalue(
        obs_rate, n_days, daily_stats, stat_func, n_bootstrap=1000, alpha=0.10
    )
    results_rate[group] = {
        'n_days': n_days,
        'obs_rate': obs_rate,
        'ci': (ci_low, ci_high),
        'p': pval,
        'sig': sig
    }

print("\n[Landfall Occurrence Rate Test Results]")
for group in groups:
    res = results_rate[group]
    print(f"Phase group {group:3s} | days: {res['n_days']:4d} | observed: {res['obs_rate']:.4f} | "
          f"90% CI: [{res['ci'][0]:.4f}, {res['ci'][1]:.4f}] | p = {res['p']:.4f} | "
          f"{'significant' if res['sig'] else 'not significant'}")

# ## 5. Part 2: Accumulated ACE

print("\n" + "="*80)
print("Part 2: Influence of MJO Phase on Accumulated ACE")
print("="*80)

results_ace = {}

for group in groups:
    group_dates = daily_stats[daily_stats['phase_group'] == group]
    n_days = len(group_dates)
    obs_ace = group_dates['sum_ACE_day'].sum()
    stat_func = lambda df: df['sum_ACE_day'].sum()
    ci_low, ci_high, pval, sig = bootstrap_ci_pvalue(
        obs_ace, n_days, daily_stats, stat_func, n_bootstrap=1000, alpha=0.10
    )
    results_ace[group] = {
        'n_days': n_days,
        'obs_ace': obs_ace,
        'ci': (ci_low, ci_high),
        'p': pval,
        'sig': sig
    }

print("\n[Accumulated ACE Test Results]")
for group in groups:
    res = results_ace[group]
    print(f"Phase group {group:3s} | days: {res['n_days']:4d} | observed: {res['obs_ace']:.2f} | "
          f"90% CI: [{res['ci'][0]:.2f}, {res['ci'][1]:.2f}] | p = {res['p']:.4f} | "
          f"{'significant' if res['sig'] else 'not significant'}")

# ## 6. Part 3: Accumulated Precipitation

print("\n" + "="*80)
print("Part 3: Influence of MJO Phase on Accumulated Precipitation")
print("="*80)

results_precip = {}

for group in groups:
    group_dates = daily_stats[daily_stats['phase_group'] == group]
    n_days = len(group_dates)
    obs_precip = group_dates['sum_precip_day'].sum()
    stat_func = lambda df: df['sum_precip_day'].sum()
    ci_low, ci_high, pval, sig = bootstrap_ci_pvalue(
        obs_precip, n_days, daily_stats, stat_func, n_bootstrap=1000, alpha=0.10
    )
    results_precip[group] = {
        'n_days': n_days,
        'obs_precip': obs_precip,
        'ci': (ci_low, ci_high),
        'p': pval,
        'sig': sig
    }

print("\n[Accumulated Precipitation Test Results]")
for group in groups:
    res = results_precip[group]
    print(f"Phase group {group:3s} | days: {res['n_days']:4d} | observed: {res['obs_precip']:.2f} | "
          f"90% CI: [{res['ci'][0]:.2f}, {res['ci'][1]:.2f}] | p = {res['p']:.4f} | "
          f"{'significant' if res['sig'] else 'not significant'}")

# ## 7. Part 4: Accumulated Precipitation by Typhoon Category

print("\n" + "="*80)
print("Part 4: Influence of MJO Phase on Accumulated Precipitation by Typhoon Intensity")
print("="*80)

categories = [1, 2, 3]
cat_names = {1: 'Cat1', 2: 'Cat2', 3: 'Cat3'}
results_precip_cat = {cat: {} for cat in categories}

for cat in categories:
    col_name = f'sum_precip_cat{cat}'
    for group in groups:
        group_dates = daily_stats[daily_stats['phase_group'] == group]
        n_days = len(group_dates)
        obs_val = group_dates[col_name].sum()
        # Use default argument to capture col_name in lambda
        stat_func = lambda df, col=col_name: df[col].sum()
        ci_low, ci_high, pval, sig = bootstrap_ci_pvalue(
            obs_val, n_days, daily_stats, stat_func, n_bootstrap=1000, alpha=0.10
        )
        results_precip_cat[cat][group] = {
            'obs': obs_val,
            'ci': (ci_low, ci_high),
            'p': pval,
            'sig': sig
        }

print("\n[Accumulated Precipitation by Category Test Results]")
for cat in categories:
    print(f"\nCategory {cat} - {cat_names[cat]}")
    for group in groups:
        res = results_precip_cat[cat][group]
        print(f"  Phase group {group:3s} | observed: {res['obs']:.2f} | "
              f"90% CI: [{res['ci'][0]:.2f}, {res['ci'][1]:.2f}] | p = {res['p']:.4f} | "
              f"{'significant' if res['sig'] else 'not significant'}")

# ## 8. Part 5: Active-Day Probability by Typhoon Category

print("\n" + "="*80)
print("Part 5: Influence of MJO Phase on Active-Day Probability by Typhoon Intensity")
print("="*80)

results_prob_cat = {cat: {} for cat in categories}

for cat in categories:
    col_name = f'has_cat{cat}'
    for group in groups:
        group_dates = daily_stats[daily_stats['phase_group'] == group]
        n_days = len(group_dates)
        obs_prob = group_dates[col_name].mean()
        stat_func = lambda df, col=col_name: df[col].mean()
        ci_low, ci_high, pval, sig = bootstrap_ci_pvalue(
            obs_prob, n_days, daily_stats, stat_func, n_bootstrap=1000, alpha=0.10
        )
        results_prob_cat[cat][group] = {
            'obs': obs_prob,
            'ci': (ci_low, ci_high),
            'p': pval,
            'sig': sig
        }

print("\n[Active-Day Probability by Category Test Results]")
for cat in categories:
    print(f"\nCategory {cat} - {cat_names[cat]}")
    for group in groups:
        res = results_prob_cat[cat][group]
        print(f"  Phase group {group:3s} | observed probability: {res['obs']:.4f} | "
              f"90% CI: [{res['ci'][0]:.4f}, {res['ci'][1]:.4f}] | p = {res['p']:.4f} | "
              f"{'significant' if res['sig'] else 'not significant'}")

Total MJO active days (1960-2024, Jun-Oct, amp≥1): 5603
Days per phase group:
1-2    1651
3-4    1000
5-6    1823
7-8    1129
Name: phase_group, dtype: int64
Landfall events meeting criteria: 258
Number of typhoons with ACE computed: 490
Number of typhoons with precipitation data loaded: 258

Part 1: Influence of MJO Phase on Landfall Occurrence Rate

[Landfall Occurrence Rate Test Results]
Phase group 1-2 | days: 1651 | observed: 0.0248 | 90% CI: [0.0376, 0.0539] | p = 1.0000 | significant
Phase group 3-4 | days: 1000 | observed: 0.0440 | 90% CI: [0.0350, 0.0570] | p = 1.0000 | not significant
Phase group 5-6 | days: 1823 | observed: 0.0647 | 90% CI: [0.0384, 0.0538] | p = 0.0000 | significant
Phase group 7-8 | days: 1129 | observed: 0.0478 | 90% CI: [0.0354, 0.0558] | p = 0.8180 | not significant

Part 2: Influence of MJO Phase on Accumulated ACE

[Accumulated ACE Test Results]
Phase group 1-2 | days: 1651 | observed: 5794098.00 | 90% CI: [9669241.43, 15544713.60] | p = 1.0000 | sign